## ***IMPORT LIBRERIE***


In [4]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR, FIGURES_DIR, MODELS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob

from models.DeepConvLSTM import DeepConvLSTM, HARDataset 
import optuna
from optuna.pruners import MedianPruner
import optuna.visualization as vis
import train
import torch
import torch.nn as nn
import train_with_cm
import matplotlib.pyplot as plt
#definisco il path da cui leggere i .csv

from utils.log_config import logger
from figures import plot_CM

#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\pdd_data'
logger.debug(path)

2025-05-05 11:22:29,971 - DEBUG - myapp - C:\codes\HumanActivityRecognition\data\pdd_data


## ***DATA PREPROCESSING***

*scansiono recording e filtro per righe non nulle*  
*OUTPUT: unico dataframe con tutte le attività non nulle*

In [7]:
#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path
# Definisco il percorso della cartella contenente i CSV

# Nome del file CSV finale
final_csv_path = os.path.join(path, 'df_ELEP_non_null.csv')

# Controllo se il file esiste già
if os.path.exists(final_csv_path):
    logger.debug(f"Il file {final_csv_path} esiste già. Lo sto caricando...")
    df_elep = pd.read_csv(final_csv_path)
else:
    suffixes = ["BE", "GE", "RE", "YE", "WE"]
    matching_files=[]
    for suffix in suffixes:
        matching_files.extend(glob.glob(os.path.join(path, f"*_{suffix}*.csv")))
        print(matching_files)

    kid_elep=[] #bambini prima del filtro
    kid_elep_no_null=[] #bambini dopo il filtro
    df_list_elep=[]

    for file in matching_files:
        print("Processing file:", file)
        df = pd.read_csv(file)
        print("Original shape:", df.shape)
        kid_elep.append(df['kid_id'].unique())
        # Filtra le righe con action_id non nullo
        df = df[df['action_id'] != 0]
        print("Filtered shape:", df.shape)
        kid_elep_no_null.append(df['kid_id'].unique())

        print("Columns:", df.columns)
        print("Action counts:\n", df['action'].value_counts())
        print("Toy counts:\n", df['toy_id'].value_counts())
        print("="*50)  # Separatore tra i file

    # mi stampo gli utenti prima di fare il merge e dopo il merge
    logger.info(f"Numero di utenti che hanno fatto almeno un azione: {len(kid_elep_no_null)/len(kid_elep)}")



    for file in matching_files:
        if not file.endswith('.csv'):
            continue
        
        #leggo solo i file che dopo l'undescore ha BE*.csv
        df_temp = pd.read_csv(file)
        df_temp = df_temp[df_temp['action_id'] != 0]  # Mantengo solo righe con attività non nulla
        df_list_elep.append(df_temp)

    # Unisci tutti i dataframe
    df_elep = pd.concat(df_list_elep)

    # Stampa informazioni sul dataframe finale
    print("Dimensioni del dataframe unificato con tutte le attività non nulle")
    print(df_elep.shape)
    print(df_elep.columns)
    print(df_elep['action'].value_counts())

    # Salva il dataframe risultante
    df_elep.to_csv(os.path.join(path, 'df_ELEP_non_null.csv'), index=False)

2025-05-05 11:23:17,529 - DEBUG - myapp - Il file C:\codes\HumanActivityRecognition\data\pdd_data\df_ELEP_non_null.csv esiste già. Lo sto caricando...


*per ogni attività trovata, salvo un .csv*  
*OUTPUT: un .csv per ogni attività non nulla (df_elep_action_11.csv, df_elep_action_19.csv ecc... )*


In [8]:
#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_elep['action_id'].unique():
    df_action = df_elep[df_elep["action_id"] == action_id] #filtro il dataframe in base all'attività
    logger.debug(f"Dimensioni del dataframe df_elep_action_{action_id} - {df_action.shape}") #log delle dimensioni del dataframe
    logger.info(f"Conteggio delle attività per df_elep_action_{action_id}") #log del conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_elep_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    logger.debug(f"Salvato il dataframe df_elep_action_{action_id}.csv")

2025-05-05 11:23:23,825 - DEBUG - myapp - Dimensioni del dataframe df_elep_action_36 - (9119, 25)
2025-05-05 11:23:23,828 - INFO - myapp - Conteggio delle attività per df_elep_action_36


2025-05-05 11:23:24,082 - DEBUG - myapp - Salvato il dataframe df_elep_action_36.csv
2025-05-05 11:23:24,086 - DEBUG - myapp - Dimensioni del dataframe df_elep_action_4 - (12744, 25)
2025-05-05 11:23:24,088 - INFO - myapp - Conteggio delle attività per df_elep_action_4
2025-05-05 11:23:24,352 - DEBUG - myapp - Salvato il dataframe df_elep_action_4.csv
2025-05-05 11:23:24,358 - DEBUG - myapp - Dimensioni del dataframe df_elep_action_3 - (16111, 25)
2025-05-05 11:23:24,359 - INFO - myapp - Conteggio delle attività per df_elep_action_3
2025-05-05 11:23:24,707 - DEBUG - myapp - Salvato il dataframe df_elep_action_3.csv
2025-05-05 11:23:24,714 - DEBUG - myapp - Dimensioni del dataframe df_elep_action_29 - (23708, 25)
2025-05-05 11:23:24,716 - INFO - myapp - Conteggio delle attività per df_elep_action_29
2025-05-05 11:23:25,186 - DEBUG - myapp - Salvato il dataframe df_elep_action_29.csv
2025-05-05 11:23:25,200 - DEBUG - myapp - Dimensioni del dataframe df_elep_action_10 - (94802, 25)
2025-0

## ***DATA PROCESSING***

*SLIDING WINDOW*  
*OUTPUT: unica matrice con tutte le windows concatenate*

In [9]:
#applico sliding window con la funzion process_csv
nb_sensor_channels = 9
sliding_window_length = 100
sliding_window_step = 50

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al giocattolo ball
#e poi concateno tutto in un unica x e y 

X, Y = [], []
kid_action_counts = {}

for action_file in [f for f in os.listdir(path) if f.endswith('.csv') and f.split('_')[1] == 'elep']:
    file_path = os.path.join(path, action_file)
    action = action_file.split('_')[-1].split('.')[0]

    X_windows, Y_windows, kid_id_action_dict = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)

    X.append(X_windows)
    Y.append(Y_windows)


    logger.info(f"Numero  totale di finestre per l'azione {action}:{len(X_windows)}")
    kid_action_counts[f"elep_action_{action}"] = kid_id_action_dict #aggiungo il dizionario al dizionario principale per tenere traccia del numero di finestre per ogni bambino per ogni azione, ogni ball_action è una chiave e il valore è un dizionario con il numero di finestre per ogni bambino
    logger.info(f"Contenuto finale di kid_action_counts: {kid_action_counts}") #per veere quante finestre per ogni azione e per ogni bambino sono state elaborte 


# Concateno tutti i dati in un unico array per X e Y
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)


# Stampo le dimensioni di X e Y
logger.info(f"Dimensioni di X finale: {X.shape}")
logger.info(f"Dimensioni di Y finale: {Y.shape}")

2025-05-05 11:23:42,205 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_elep_action_10.csv
2025-05-05 11:23:42,230 - INFO - myapp - Kid_id: 3003, X_kid shape: (8637, 9), Y_kid shape: (8637,)
2025-05-05 11:23:42,236 - INFO - myapp - Numero di finestre estratte: 171
2025-05-05 11:23:42,238 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 37
2025-05-05 11:23:42,242 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-05 11:23:42,243 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-05 11:23:42,244 - INFO - myapp - Numero totale di finestre (dopo padding finale): 172
2025-05-05 11:23:42,245 - DEBUG - myapp - X_windows shape after sliding window: (172, 100, 9)
2025-05-05 11:23:42,246 - DEBUG - myapp - Padding codes: [1]
2025-05-05 11:23:42,247 - INFO - myapp - Numero di finestre estratte: 171
2025-05-05 11:23:42,249 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 37
2025-

>>> Kid_ids: [3003 3007 3008 3010 3011 3013 3017 3002 3009 3019 3022 3020 3024]


2025-05-05 11:23:42,394 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 35
2025-05-05 11:23:42,396 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-05 11:23:42,399 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-05 11:23:42,400 - INFO - myapp - Numero totale di finestre (dopo padding finale): 183
2025-05-05 11:23:42,402 - DEBUG - myapp - X_windows shape after sliding window: (183, 100, 9)
2025-05-05 11:23:42,403 - DEBUG - myapp - Padding codes: [1]
2025-05-05 11:23:42,404 - INFO - myapp - Numero di finestre estratte: 182
2025-05-05 11:23:42,408 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 35
2025-05-05 11:23:42,410 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-05 11:23:42,411 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 1)
2025-05-05 11:23:42,413 - INFO - myapp - Numero totale di finestre (dopo padding finale): 183
2025-05-05 11:23:42,414 - DEBUG - myapp - Y_windows_full 

>>> Kid_ids: [3010 3007 3009 3011 3006 3023]
>>> Kid_ids: [3003 3007 3011 3017 3008 3022 3019]


2025-05-05 11:23:42,849 - INFO - myapp - Kid_id: 3003, X_kid shape: (131, 9), Y_kid shape: (131,)
2025-05-05 11:23:42,850 - INFO - myapp - Numero di finestre estratte: 1
2025-05-05 11:23:42,851 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 31
2025-05-05 11:23:42,852 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-05 11:23:42,853 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-05 11:23:42,855 - INFO - myapp - Numero totale di finestre (dopo padding finale): 2
2025-05-05 11:23:42,856 - DEBUG - myapp - X_windows shape after sliding window: (2, 100, 9)
2025-05-05 11:23:42,858 - DEBUG - myapp - Padding codes: [1]
2025-05-05 11:23:42,859 - INFO - myapp - Numero di finestre estratte: 1
2025-05-05 11:23:42,860 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 31
2025-05-05 11:23:42,860 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-05 11:23:42,861 - DEBUG - myapp - Nuova finestra con padding:

...

>>> Kid_ids: [3010 3009]


2025-05-05 11:23:47,537 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_elep_action_8.csv
2025-05-05 11:23:47,547 - INFO - myapp - Kid_id: 3003, X_kid shape: (14953, 9), Y_kid shape: (14953,)
2025-05-05 11:23:47,548 - INFO - myapp - Numero di finestre estratte: 298
2025-05-05 11:23:47,550 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 3
2025-05-05 11:23:47,551 - INFO - myapp - Non è stato applicato padding perché i campioni rimanenti sono troppo pochi.
2025-05-05 11:23:47,553 - INFO - myapp - Numero totale di finestre (dopo padding finale): 298
2025-05-05 11:23:47,553 - DEBUG - myapp - X_windows shape after sliding window: (298, 100, 9)
2025-05-05 11:23:47,555 - DEBUG - myapp - Padding codes: []
2025-05-05 11:23:47,556 - INFO - myapp - Numero di finestre estratte: 298
2025-05-05 11:23:47,559 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 3
2025-05-05 11:23:47,561 - INFO - myapp - Non è stato ap

>>> Kid_ids: [3003 3010 3011 3018 3019 3022 3023 3020 3013]


2025-05-05 11:23:47,723 - INFO - myapp - Padding estremo applicato. Padding code: 2
2025-05-05 11:23:47,724 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-05 11:23:47,727 - INFO - myapp - Numero totale di finestre (dopo padding finale): 32
2025-05-05 11:23:47,729 - DEBUG - myapp - X_windows shape after sliding window: (32, 100, 9)
2025-05-05 11:23:47,730 - DEBUG - myapp - Padding codes: [2]
2025-05-05 11:23:47,733 - INFO - myapp - Numero di finestre estratte: 31
2025-05-05 11:23:47,734 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 11
2025-05-05 11:23:47,737 - INFO - myapp - Padding estremo applicato. Padding code: 2
2025-05-05 11:23:47,738 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 1)
2025-05-05 11:23:47,740 - INFO - myapp - Numero totale di finestre (dopo padding finale): 32
2025-05-05 11:23:47,742 - DEBUG - myapp - Y_windows_full shape after sliding window: (32, 100, 1)
2025-05-05 11:23:47,744 - DEBUG - myapp - Padding codes: [2]

>>> Kid_ids: [3010 3009 3024]


In [10]:
from collections import defaultdict

kid_summary = defaultdict(dict)

for action, kid_counts in kid_action_counts.items():
    for kid, count in kid_counts.items():
        kid_summary[kid][action] = count

# Stampo il riepilogo per ogni bambino
for kid, actions in kid_summary.items():
    logger.info(f"Bambino {kid}:")
    for action, count in actions.items():
        logger.info(f"  {action}: {count} finestre")
    logger.info("="*50)

2025-05-05 11:24:06,883 - INFO - myapp - Bambino 3003:
2025-05-05 11:24:06,886 - INFO - myapp -   elep_action_10: 172 finestre
2025-05-05 11:24:06,887 - INFO - myapp -   elep_action_12: 2 finestre
2025-05-05 11:24:06,889 - INFO - myapp -   elep_action_16: 4 finestre
2025-05-05 11:24:06,891 - INFO - myapp -   elep_action_19: 14 finestre
2025-05-05 11:24:06,892 - INFO - myapp -   elep_action_21: 110 finestre
2025-05-05 11:24:06,894 - INFO - myapp -   elep_action_27: 11 finestre
2025-05-05 11:24:06,896 - INFO - myapp -   elep_action_29: 422 finestre
2025-05-05 11:24:06,898 - INFO - myapp -   elep_action_3: 19 finestre
2025-05-05 11:24:06,900 - INFO - myapp -   elep_action_31: 3 finestre
2025-05-05 11:24:06,901 - INFO - myapp -   elep_action_36: 83 finestre
2025-05-05 11:24:06,903 - INFO - myapp -   elep_action_4: 98 finestre
2025-05-05 11:24:06,904 - INFO - myapp -   elep_action_41: 14 finestre
2025-05-05 11:24:06,907 - INFO - myapp -   elep_action_6: 9 finestre
2025-05-05 11:24:06,909 - 

*SPLIT DATASET IN TRS, VS,TS*  
*OUTPUT: array NumPy di TRS/VS/TS (X e Y)*

*RIASSEGNAZIONE DELLE ETICHETTE*

## ***OPTUNA***

*nella funzione obiettivo: uso train e val*  
*stampo cm di train e val*  
*train finale con dati di test con cm*


